In [1]:
!pip uninstall -y -q torchaudio torchcodec
!pip install -q vllm
!python -c "import torch, vllm; print('torch', torch.__version__, '| vllm', vllm.__version__, '| GPU', torch.cuda.device_count())"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/

In [2]:
import os, pandas as pd, numpy as np

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]

tr = pd.read_csv(f"{COMP}/train.csv")
gold = tr[tr[LABELS].notna().all(axis=1)][["StudyInstanceUID"] + LABELS]
gold_rep = (gold[["StudyInstanceUID"]]
            .merge(tr[["StudyInstanceUID", "Report"]], on="StudyInstanceUID", how="inner")
            .rename(columns={"Report": "report"}))
gold_rep.to_csv("/kaggle/working/gold_reports.csv", index=False)

lens = gold_rep["report"].astype(str).str.len()
print(f"Studi gold dengan laporan: {len(gold_rep)}/{len(gold)}")
print(f"Panjang laporan (karakter): median {int(lens.median())}, maks {int(lens.max())}, "
      f"terpotong di 8000: {int((lens > 8000).sum())} laporan")

Studi gold dengan laporan: 58/58
Panjang laporan (karakter): median 1205, maks 3891, terpotong di 8000: 0 laporan


In [3]:
%%writefile /kaggle/working/label_llm.py
import os, re, json, argparse
import pandas as pd
from vllm import LLM, SamplingParams

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]
DEFS = {
    "ACL": "anterior cruciate ligament tear or injury (partial or complete, including tear of an ACL graft)",
    "MCL": "medial collateral ligament sprain, tear or injury",
    "Medial Meniscus": "tear of the medial meniscus",
    "Lateral Meniscus": "tear of the lateral meniscus",
    "Medial OA": "osteoarthritis or degenerative cartilage loss of the medial tibiofemoral compartment",
    "Lateral OA": "osteoarthritis or degenerative cartilage loss of the lateral tibiofemoral compartment",
    "PF OA": "osteoarthritis or degenerative cartilage loss of the patellofemoral compartment",
    "Effusion": "joint effusion",
    "Synovitis": "synovitis or synovial thickening/inflammation",
    "Baker's": "Baker's (popliteal) cyst",
    "Contusion": "bone contusion, bone bruise or traumatic bone marrow edema",
    "Fracture": "fracture of any bone of the knee (including avulsion or insufficiency fracture)",
}
PROMPT = """You are an expert musculoskeletal radiologist. Read the knee MRI report below. It may be written in any language.

For each finding, answer exactly one of "positive", "negative" or "uncertain":
- positive: the finding is clearly present, at any severity (including mild, small or partial).
- uncertain: the report says possible, suspected, cannot be excluded, or is equivocal.
- negative: the report states it is absent or normal, OR the finding is not mentioned.

Findings:
{defs}

Report:
\"\"\"
{report}
\"\"\"

Answer with a single JSON object only, using exactly these keys: {keys}"""

def to_num(v):
    v = str(v).strip().lower()
    if v.startswith("pos"): return 1.0
    if v.startswith("unc"): return 0.5
    if v.startswith("neg"): return 0.0
    return float("nan")

def parse(text):
    m = re.search(r"\{.*\}", text, re.S)
    out = {l: float("nan") for l in LABELS}
    if m:
        try:
            d = json.loads(m.group(0))
            for l in LABELS:
                if l in d: out[l] = to_num(d[l])
            return out
        except Exception:
            pass
    for l in LABELS:   # cadangan: cari pola "Label": "nilai"
        mm = re.search(re.escape(l) + r"\W+(positive|negative|uncertain)", text, re.I)
        if mm: out[l] = to_num(mm.group(1))
    return out

ap = argparse.ArgumentParser()
ap.add_argument("--model"); ap.add_argument("--out"); ap.add_argument("--quant", default=None)
a = ap.parse_args()

df = pd.read_csv("/kaggle/working/gold_reports.csv")
llm = LLM(model=a.model, quantization=a.quant, dtype="float16", tensor_parallel_size=2,
          max_model_len=6144, gpu_memory_utilization=0.90, enforce_eager=True,
          disable_custom_all_reduce=True)
tok = llm.get_tokenizer()
defs = "\n".join(f'- "{l}": {DEFS[l]}' for l in LABELS)
keys = ", ".join(f'"{l}"' for l in LABELS)
prompts = [tok.apply_chat_template(
              [{"role": "user", "content": PROMPT.format(defs=defs, report=str(r)[:8000], keys=keys)}],
              tokenize=False, add_generation_prompt=True) for r in df["report"]]
outs = llm.generate(prompts, SamplingParams(temperature=0, max_tokens=400))

rows = []
for uid, o in zip(df["StudyInstanceUID"], outs):
    txt = o.outputs[0].text
    rows.append({"StudyInstanceUID": uid, **parse(txt), "raw": txt})
res = pd.DataFrame(rows)
res.to_csv(a.out, index=False)
print(f"{a.model}: selesai {len(res)} laporan | sel kosong: {int(res[LABELS].isna().sum().sum())}")
os._exit(0)

Writing /kaggle/working/label_llm.py


In [4]:
%%bash
cd /kaggle/working
run () {
  name=$1; shift
  echo "=== $name mulai $(date +%T)"
  setsid timeout 45m python label_llm.py "$@" > log_$name.txt 2>&1 &
  pid=$!
  wait $pid
  echo "kode keluar: $?"
  kill -9 -- -$pid 2>/dev/null      
  sleep 10
  tail -n 3 log_$name.txt
  nvidia-smi --query-gpu=memory.used --format=csv,noheader
  echo "=== $name selesai $(date +%T)"
}
run 7b  --model Qwen/Qwen2.5-7B-Instruct --out labels_7b.csv
run 14b --model Qwen/Qwen2.5-14B-Instruct-AWQ --quant awq --out labels_14b.csv
run 32b --model Qwen/Qwen2.5-32B-Instruct-AWQ --quant awq --out labels_32b.csv
ls -la labels_*.csv

=== 7b mulai 02:05:00
kode keluar: 0
(MainProcess pid=180) INFO 10-10 02:08:01 [hf.py:646] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
Processed prompts: 100%|██████████| 58/58 [00:31<00:00,  1.83it/s, est. speed input: 1562.70 toks/s, output: 203.30 toks/s]
0 MiB
0 MiB
=== 7b selesai 02:08:43
=== 14b mulai 02:08:43
kode keluar: 0
(MainProcess pid=546) INFO 10-10 02:10:52 [hf.py:646] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
Processed prompts: 100%|██████████| 58/58 [01:03<00:00,  1.09s/it, est. speed input: 780.30 toks/s, output: 100.52 toks/s]
0 MiB
0 MiB
=== 14b selesai 02:12:06
=== 32b mulai 02:12:06
kode keluar: 0
(MainProcess pid=819) INFO 10-10 02:16:01 [hf.py:646] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
Processed prompts: 100%|██████████| 58/58 

In [5]:
from sklearn.metrics import roc_auc_score

def find_file(name):
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d != "competitions"]
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)

def per_label_auc(y, p):
    aucs = []
    for k in range(len(LABELS)):
        ok = ~np.isnan(y[:, k]) & (y[:, k] != 0.5) & ~np.isnan(p[:, k])
        yk, pk = y[ok, k], p[ok, k]
        aucs.append(roc_auc_score(yk, pk) if len(np.unique(yk)) == 2 else np.nan)
    return np.array(aucs)

ids = gold_rep["StudyInstanceUID"]
Y = gold.set_index("StudyInstanceUID").loc[ids, LABELS].values.astype(float)
methods = {"lama (7B, prompt lama)": pd.read_csv(find_file("report_labels.csv"))}
for tag, f in [("7B", "labels_7b.csv"), ("14B-AWQ", "labels_14b.csv"), ("32B-AWQ", "labels_32b.csv")]:
    if os.path.exists(f"/kaggle/working/{f}"):
        methods[tag] = pd.read_csv(f"/kaggle/working/{f}")
P = {m: d.set_index("StudyInstanceUID").reindex(ids)[LABELS].values.astype(float) for m, d in methods.items()}

table = pd.DataFrame({m: per_label_auc(Y, p) for m, p in P.items()}, index=LABELS)
table.loc["RATA-RATA"] = table.mean()
print(table.round(3).to_string())

# bootstrap: seberapa yakin selisihnya dibanding label lama (tidak kebetulan)?
rng = np.random.default_rng(0)
base = "lama (7B, prompt lama)"
for m in P:
    if m == base: continue
    diffs = []
    for _ in range(1000):
        i = rng.integers(0, len(ids), len(ids))
        diffs.append(np.nanmean(per_label_auc(Y[i], P[m][i])) - np.nanmean(per_label_auc(Y[i], P[base][i])))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    print(f"{m:10s} vs label lama: selisih {np.mean(diffs):+.3f} (95% CI {lo:+.3f} .. {hi:+.3f})")

                  lama (7B, prompt lama)     7B  14B-AWQ  32B-AWQ
ACL                                0.890  0.869    0.896    0.937
MCL                                0.939  0.908    0.908    0.918
Medial Meniscus                    0.867  0.895    0.865    0.916
Lateral Meniscus                   0.771  0.791    0.793    0.845
Medial OA                          0.863  0.917    0.865    0.909
Lateral OA                         0.810  0.867    0.802    0.854
PF OA                              0.782  0.626    0.689    0.781
Effusion                           0.677  0.763    0.665    0.696
Synovitis                          0.674  0.704    0.726    0.758
Baker's                            0.893  0.893    0.900    0.892
Contusion                          0.753  0.731    0.727    0.742
Fracture                           0.792  0.778    0.772    0.767
RATA-RATA                          0.809  0.812    0.801    0.835
7B         vs label lama: selisih +0.003 (95% CI -0.019 .. +0.027)
14B-AWQ  